In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [28]:
!nvidia-smi

Mon Feb 23 19:22:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   63C    P0             30W /   72W |   21304MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
REPO_URL = "https://github.com/SpaceSpiff20/daniel-splits.git"
REPO_DIR = "daniel-splits"

In [4]:
!git clone {REPO_URL}
%cd {REPO_DIR}

Cloning into 'daniel-splits'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 64 (delta 21), reused 57 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 319.89 KiB | 1.78 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/daniel-splits


In [5]:
# switch to daniel_splits branch
!git checkout daniel_splits
!git pull

Branch 'daniel_splits' set up to track remote branch 'daniel_splits' from 'origin'.
Switched to a new branch 'daniel_splits'
Already up to date.


In [6]:
!ls data/splits_out/superglue_boolq

manifest.json	  N16_seed0.jsonl   N256_seed1.jsonl  N32_seed2.jsonl
N128_seed0.jsonl  N16_seed1.jsonl   N256_seed2.jsonl  N64_seed0.jsonl
N128_seed1.jsonl  N16_seed2.jsonl   N32_seed0.jsonl   N64_seed1.jsonl
N128_seed2.jsonl  N256_seed0.jsonl  N32_seed1.jsonl   N64_seed2.jsonl


In [7]:
# Install libraries
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip -q install transformers datasets accelerate peft evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [8]:
from pathlib import Path

SPLITS_DIR = Path("data/splits_out/superglue_boolq")

assert SPLITS_DIR.exists(), f"Missing splits dir: {SPLITS_DIR}"
print("Found split files:", sorted([p.name for p in SPLITS_DIR.glob("*.jsonl")])[:10])

Found split files: ['N128_seed0.jsonl', 'N128_seed1.jsonl', 'N128_seed2.jsonl', 'N16_seed0.jsonl', 'N16_seed1.jsonl', 'N16_seed2.jsonl', 'N256_seed0.jsonl', 'N256_seed1.jsonl', 'N256_seed2.jsonl', 'N32_seed0.jsonl']


In [9]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
!pip -q install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.2 MB/s eta 0:00:00


In [11]:
# Load Qwen3-4B-Instruct
!pip -q install -U bitsandbytes accelerate transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="cuda:0",            # keep everything on the single T4
    quantization_config=bnb_config, # <-- this is what actually reduces VRAM
    attn_implementation="eager",
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Optional: reduce KV-cache memory spikes for long prompts
model.config.use_cache = False

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 71.8 MB/s eta 0:00:00


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

In [12]:
import re
from datasets import load_dataset
from pathlib import Path
import torch

In [13]:
SPLITS_DIR = Path("data/splits_out/superglue_boolq")


In [14]:
# Parse your JSONL 'x' field:
# "Passage: ...\nQuestion: ..."
X_RE = re.compile(r"^Passage:\s*(?P<passage>.*)\nQuestion:\s*(?P<question>.*)$", re.DOTALL)

In [15]:
def parse_x(x: str):
    m = X_RE.match(x.strip())
    if not m:
        # Fallback: do something reasonable if formatting drifts
        # Try splitting on "\nQuestion:"
        parts = x.split("\nQuestion:")
        if len(parts) == 2:
            passage = parts[0].replace("Passage:", "").strip()
            question = parts[1].strip()
            return passage, question
        raise ValueError(f"Could not parse x field:\n{x[:200]}...")
    return m.group("passage").strip(), m.group("question").strip()

In [16]:
def label_to_yesno(label):
    s = str(label).strip().lower()
    if s in {"true", "1", "yes"}:
        return "yes"
    if s in {"false", "0", "no"}:
        return "no"
    raise ValueError(f"Unknown label: {label}")

In [17]:
def build_messages_from_demos(demo_rows, passage, question, k: int):
    msgs = [
        {"role": "system", "content": "Answer with exactly one token: yes or no."}
    ]

    for ex in demo_rows[:k]:
        dp, dq = parse_x(ex["x"])
        msgs.append({"role": "user", "content": f"Passage:\n{dp}\n\nQuestion: {dq}\nAnswer (yes/no):"})
        msgs.append({"role": "assistant", "content": label_to_yesno(ex["label"])})

    msgs.append({"role": "user", "content": f"Passage:\n{passage}\n\nQuestion: {question}\nAnswer (yes/no):"})
    return msgs

In [18]:
YES_RE = re.compile(r"\byes\b", re.IGNORECASE)
NO_RE  = re.compile(r"\bno\b", re.IGNORECASE)

In [19]:
import time
import numpy as np

In [20]:
@torch.inference_mode()
def generate_and_measure(prompt_text, max_new_tokens=3):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    prompt_tokens = int(inputs["input_ids"].shape[1])

    # peak memory + generation-only timing
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    gen_ids = out[0][prompt_tokens:]
    gen_tokens = int(gen_ids.numel())
    gen_time_s = (t1 - t0)
    toks_per_sec = (gen_tokens / gen_time_s) if gen_time_s > 0 else float("inf")

    peak_mb = None
    if torch.cuda.is_available():
        peak_mb = float(torch.cuda.max_memory_allocated() / (1024**2))

    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return gen_text, {
        "prompt_tokens": prompt_tokens,
        "gen_tokens": gen_tokens,
        "gen_time_s": gen_time_s,
        "toks_per_sec": toks_per_sec,
        "peak_mb": peak_mb,
    }

Old version

In [21]:
# @title
from datasets import load_dataset

def run_boolq_icl(N: int, seed: int, max_eval=200, max_new_tokens=3):
    demo_path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
    demos = load_dataset("json", data_files=str(demo_path), split="train")

    # Evaluate on held-out validation
    val = load_dataset("super_glue", "boolq", split="validation")
    if max_eval is not None:
        val = val.select(range(min(len(val), max_eval)))

    correct = 0
    invalid = 0
    lat_s = []
    prompt_toks = []
    gen_toks = []
    tps = []
    peaks = []

    demo_rows = list(demos)  # small enough for N<=256

    # Warmup
    wp = tokenizer.apply_chat_template(
        build_messages_from_demos(demo_rows, val[0]["passage"], val[0]["question"], k=min(N, len(demo_rows))),
        tokenize=False,
        add_generation_prompt=True,
    )
    _ = generate_and_measure(wp, max_new_tokens=max_new_tokens)

    for ex in val:
        gold = "yes" if ex["label"] else "no"  # SuperGLUE BoolQ validation uses boolean label

        msgs = build_messages_from_demos(demo_rows, ex["passage"], ex["question"], k=min(N, len(demo_rows)))
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

        gen_text, m = generate_and_measure(prompt, max_new_tokens=max_new_tokens)
        pred = parse_model_yesno(gen_text)

        if pred == "unknown":
            invalid += 1
        if pred == gold:
            correct += 1

        lat_s.append(m["gen_time_s"])
        prompt_toks.append(m["prompt_tokens"])
        gen_toks.append(m["gen_tokens"])
        tps.append(m["toks_per_sec"])
        if m["peak_mb"] is not None:
            peaks.append(m["peak_mb"])

    n = len(val)
    return {
        "N": N,
        "seed": seed,
        "n_eval": n,
        "accuracy": correct / n if n else 0.0,
        "invalid_frac": invalid / n if n else 0.0,
        "gen_time_s_mean": float(np.mean(lat_s)),
        "toks_per_sec_mean": float(np.mean(tps)),
        "prompt_tokens_mean": float(np.mean(prompt_toks)),
        "prompt_tokens_max": int(np.max(prompt_toks)) if prompt_toks else 0,
        "peak_mb_max": float(np.max(peaks)) if peaks else None,
    }

Dynamic K

In [29]:
# @title
from datasets import load_dataset
import numpy as np
import gc
import torch

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_boolq_icl(
    N: int,
    seed: int,
    max_eval=200,
    max_new_tokens=3,
    demo_chars=250,            # truncate DEMO passages only (big OOM reduction)
    max_prompt_tokens=6000,    # prompt budget; will back off k if exceeded
    cleanup_every=25,          # periodic cleanup to prevent fragmentation creep
):
    demo_path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
    demos = load_dataset("json", data_files=str(demo_path), split="train")
    demo_rows = list(demos)  # N<=256 so fine

    # Evaluate on held-out validation
    val = load_dataset("super_glue", "boolq", split="validation")
    if max_eval is not None:
        val = val.select(range(min(len(val), max_eval)))

    # ---- warmup: tiny prompt, NOT k-shot ----
    warm_msgs = [
        {"role": "system", "content": "Answer with exactly one token: yes or no."},
        {"role": "user", "content": "Passage:\nDogs are mammals.\n\nQuestion: are dogs mammals?\nAnswer (yes/no):"},
    ]
    warm_prompt = tokenizer.apply_chat_template(warm_msgs, tokenize=False, add_generation_prompt=True)
    _ = generate_and_measure(warm_prompt, max_new_tokens=max_new_tokens)
    cleanup_cuda()

    correct = 0
    invalid = 0
    skipped = 0
    oom = 0

    lat_s = []
    prompt_toks = []
    gen_toks = []
    tps = []
    peaks = []
    k_used_list = []

    def build_messages_from_demos_trunc(demo_rows, passage, question, k: int):
        msgs = [{"role": "system", "content": "Answer with exactly one token: yes or no."}]
        for ex in demo_rows[:k]:
            dp, dq = parse_x(ex["x"])
            dp = dp[:demo_chars]  # demos only
            msgs.append({"role": "user", "content": f"Passage:\n{dp}\n\nQuestion: {dq}\nAnswer (yes/no):"})
            msgs.append({"role": "assistant", "content": label_to_yesno(ex["label"])})
        msgs.append({"role": "user", "content": f"Passage:\n{passage}\n\nQuestion: {question}\nAnswer (yes/no):"})
        return msgs

    def prompt_with_budget(passage, question, k_init):
        k = min(k_init, len(demo_rows))
        while True:
            msgs = build_messages_from_demos_trunc(demo_rows, passage, question, k=k)
            prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            # token-count check BEFORE we try to allocate big attention buffers
            tok = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
            if tok <= max_prompt_tokens or k == 0:
                return prompt, k, int(tok)
            # back off aggressively to fit budget
            k = k // 2

    for i, ex in enumerate(val):
        gold = "yes" if ex["label"] else "no"

        prompt, k_used, _tok = prompt_with_budget(ex["passage"], ex["question"], k_init=N)
        k_used_list.append(k_used)

        try:
            gen_text, m = generate_and_measure(prompt, max_new_tokens=max_new_tokens)
        except torch.cuda.OutOfMemoryError:
            oom += 1
            cleanup_cuda()
            # If we still OOM even after budget/backoff, skip this example
            skipped += 1
            continue

        pred = parse_model_yesno(gen_text)

        if pred == "unknown":
            invalid += 1
        if pred == gold:
            correct += 1

        lat_s.append(m["gen_time_s"])
        prompt_toks.append(m["prompt_tokens"])
        gen_toks.append(m["gen_tokens"])
        tps.append(m["toks_per_sec"])
        if m["peak_mb"] is not None:
            peaks.append(m["peak_mb"])

        # periodic cleanup to avoid fragmentation creep
        if cleanup_every and (i + 1) % cleanup_every == 0:
            cleanup_cuda()

    n_total = len(val)
    n_used = len(lat_s)  # evaluated examples that didn’t get skipped/oom

    return {
        "N": N,
        "seed": seed,
        "n_eval_total": n_total,
        "n_eval_used": n_used,
        "skipped": skipped,
        "oom": oom,
        "accuracy": (correct / n_used) if n_used else 0.0,
        "invalid_frac": (invalid / n_used) if n_used else 0.0,
        "gen_time_s_mean": float(np.mean(lat_s)) if lat_s else None,
        "toks_per_sec_mean": float(np.mean(tps)) if tps else None,
        "prompt_tokens_mean": float(np.mean(prompt_toks)) if prompt_toks else None,
        "prompt_tokens_max": int(np.max(prompt_toks)) if prompt_toks else 0,
        "peak_mb_max": float(np.max(peaks)) if peaks else None,
        "k_used_mean": float(np.mean(k_used_list)) if k_used_list else None,
        "k_used_min": int(np.min(k_used_list)) if k_used_list else None,
    }

No dynamic K

In [35]:
from datasets import load_dataset
import numpy as np
import gc
import torch

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def run_boolq_icl(
    N: int,
    seed: int,
    max_eval=200,
    max_new_tokens=3,
    demo_chars=250,          # truncate DEMO passages only
    max_prompt_tokens=12000,  # if exceeded, skip example (no k backoff)
    cleanup_every=25,
):
    demo_path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
    demos = load_dataset("json", data_files=str(demo_path), split="train")
    demo_rows = list(demos)

    val = load_dataset("super_glue", "boolq", split="validation")
    if max_eval is not None:
        val = val.select(range(min(len(val), max_eval)))

    # Tiny warmup (not N-shot)
    warm_msgs = [
        {"role": "system", "content": "Answer with exactly one token: yes or no."},
        {"role": "user", "content": "Passage:\nDogs are mammals.\n\nQuestion: are dogs mammals?\nAnswer (yes/no):"},
    ]
    warm_prompt = tokenizer.apply_chat_template(warm_msgs, tokenize=False, add_generation_prompt=True)
    _ = generate_and_measure(warm_prompt, max_new_tokens=max_new_tokens)
    cleanup_cuda()

    correct = 0
    invalid = 0
    skipped = 0
    oom = 0

    lat_s = []
    prompt_toks = []
    gen_toks = []
    tps = []
    peaks = []

    def build_messages_from_demos_trunc(passage, question, k: int):
        msgs = [{"role": "system", "content": "Answer with exactly one token: yes or no."}]
        for ex in demo_rows[:k]:
            dp, dq = parse_x(ex["x"])
            dp = dp[:demo_chars]  # demos only
            msgs.append({"role": "user", "content": f"Passage:\n{dp}\n\nQuestion: {dq}\nAnswer (yes/no):"})
            msgs.append({"role": "assistant", "content": label_to_yesno(ex["label"])})
        msgs.append({"role": "user", "content": f"Passage:\n{passage}\n\nQuestion: {question}\nAnswer (yes/no):"})
        return msgs

    for i, ex in enumerate(val):
        gold = "yes" if ex["label"] else "no"
        k_fixed = min(N, len(demo_rows))

        msgs = build_messages_from_demos_trunc(ex["passage"], ex["question"], k=k_fixed)
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

        # Token budget guard (skip, don’t change k)
        tok = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
        if tok > max_prompt_tokens:
            skipped += 1
            continue

        try:
            gen_text, m = generate_and_measure(prompt, max_new_tokens=max_new_tokens)
        except torch.cuda.OutOfMemoryError:
            oom += 1
            cleanup_cuda()
            skipped += 1
            continue

        pred = parse_model_yesno(gen_text)
        if pred == "unknown":
            invalid += 1
        if pred == gold:
            correct += 1

        lat_s.append(m["gen_time_s"])
        prompt_toks.append(m["prompt_tokens"])
        gen_toks.append(m["gen_tokens"])
        tps.append(m["toks_per_sec"])
        if m["peak_mb"] is not None:
            peaks.append(m["peak_mb"])

        if cleanup_every and (i + 1) % cleanup_every == 0:
            cleanup_cuda()

    n_total = len(val)
    n_used = len(lat_s)

    return {
        "N": N,
        "seed": seed,
        "n_eval_total": n_total,
        "n_eval_used": n_used,
        "skipped": skipped,
        "oom": oom,
        "accuracy": (correct / n_used) if n_used else 0.0,
        "invalid_frac": (invalid / n_used) if n_used else 0.0,
        "gen_time_s_mean": float(np.mean(lat_s)) if lat_s else None,
        "toks_per_sec_mean": float(np.mean(tps)) if tps else None,
        "prompt_tokens_mean": float(np.mean(prompt_toks)) if prompt_toks else None,
        "prompt_tokens_max": int(np.max(prompt_toks)) if prompt_toks else 0,
        "peak_mb_max": float(np.max(peaks)) if peaks else None,
    }

In [22]:
import pandas as pd

In [23]:
def parse_model_yesno(text: str):
    t = text.strip().lower()
    # accept "yes." / "no." / "yes\n..."
    if t.startswith("yes"):
        return "yes"
    if t.startswith("no"):
        return "no"

    has_yes = YES_RE.search(t) is not None
    has_no  = NO_RE.search(t) is not None

    if has_yes and not has_no:
        return "yes"
    if has_no and not has_yes:
        return "no"
    return "unknown"

In [36]:
# Smoke test
r = run_boolq_icl(
    N=64,
    seed=0,
    max_eval=1,
    max_new_tokens=3
)
print(r)

{'N': 64, 'seed': 0, 'n_eval_total': 1, 'n_eval_used': 1, 'skipped': 0, 'oom': 0, 'accuracy': 1.0, 'invalid_frac': 0.0, 'gen_time_s_mean': 8.318561947000035, 'toks_per_sec_mean': 0.24042617134338587, 'prompt_tokens_mean': 6045.0, 'prompt_tokens_max': 6045, 'peak_mb_max': 15528.2919921875}


In [ ]:
# Debug one example end-to-end

N = 16
seed = 0

demo_path = SPLITS_DIR / f"N{N}_seed{seed}.jsonl"
demos = load_dataset("json", data_files=str(demo_path), split="train")
val = load_dataset("super_glue", "boolq", split="validation")

ex = val[0]
demo_rows = list(demos)

msgs = build_messages_from_demos(
    demo_rows,
    ex["passage"],
    ex["question"],
    k=N
)

prompt = tokenizer.apply_chat_template(
    msgs,
    tokenize=False,
    add_generation_prompt=True
)

print("==== PROMPT ====")
print(prompt[:2000])  # avoid flooding notebook

gen_text, metrics = generate_and_measure(prompt)
print("\n==== RAW OUTPUT ====")
print(gen_text)
print("\n==== PARSED ====")
print(parse_model_yesno(gen_text))
print("\n==== METRICS ====")
print(metrics)

==== PROMPT ====
<|im_start|>system
Answer with exactly one token: yes or no.<|im_end|>
<|im_start|>user
Passage:
Buttermilk pie -- Buttermilk pie is a custard-like pie. Originally from the United Kingdom, it is now a traditional pie of the southern United States. It is similar to, and sometimes confused with, chess pie but it does not include cornmeal. The basic filling consists of a mixture of sugar, butter, eggs, buttermilk and wheat flour. Variations on the recipe may include flavorings such as vanilla, lemon zest and nutmeg. Buttermilk pies are made with a pie crust. The filling is poured into the crust and baked until the mixture sets. The pie is best eaten at room temperature after being allowed to cool, but may be eaten either warm from the oven or after being chilled.

Question: are chess pie and buttermilk pie the same
Answer (yes/no):<|im_end|>
<|im_start|>assistant
no<|im_end|>
<|im_start|>user
Passage:
Hyundai Genesis Coupe -- The Hyundai Genesis Coupé is a rear-wheel driv

In [47]:
# prompt explosion test
r = run_boolq_icl(
    N=128,
    seed=0,
    max_eval=2,
    max_new_tokens=3
)
print(r)

Generating train split: 0 examples [00:00, ? examples/s]

{'N': 128, 'seed': 0, 'n_eval_total': 2, 'n_eval_used': 0, 'skipped': 2, 'oom': 2, 'accuracy': 0.0, 'invalid_frac': 0.0, 'gen_time_s_mean': None, 'toks_per_sec_mean': None, 'prompt_tokens_mean': None, 'prompt_tokens_max': 0, 'peak_mb_max': None}


In [42]:
import gc, torch

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [44]:
import pandas as pd

df_partial = pd.read_csv("qwen3_boolq_icl_eval.csv")
df_partial

,N,seed,n_eval,accuracy,invalid_frac,gen_time_s_mean,toks_per_sec_mean,prompt_tokens_mean,prompt_tokens_max,peak_mb_max
0,16,0,200,0.885,0.0,5.264400,0.380604,2871.88,3450,7818.600586
1,16,1,200,0.885,0.0,5.389296,0.371676,2921.88,3500,7934.274414
2,16,2,200,0.890,0.0,4.388051,0.457151,2574.88,3153,7163.149902
3,32,0,200,0.890,0.0,7.266327,0.275787,5665.88,6244,15630.029300
4,32,1,200,0.880,0.0,6.275121,0.319204,5250.88,5829,14017.068360
5,32,2,200,0.890,0.0,6.859340,0.292135,5495.88,6074,14956.506350


In [38]:
import pandas as pd
# Ns = [16, 32, 64, 128, 256]
Ns = [64]
seeds = [0, 1, 2]

rows = []
for N in Ns:
    for seed in seeds:
        cleanup_cuda()  # <-- before starting a new (N, seed)
        r = run_boolq_icl(N=N, seed=seed, max_eval=50, max_new_tokens=3)
        print(r)
        rows.append(r)
        cleanup_cuda()  # <-- after finishing the run

df = pd.DataFrame(rows).sort_values(["N", "seed"])
df

KeyboardInterrupt: 

In [46]:
import pandas as pd

Ns = [64]
seeds = [0, 1, 2]

rows_new = []
for N in Ns:
    for seed in seeds:
        cleanup_cuda()  # <-- before starting a new (N, seed)
        r = run_boolq_icl(N=N, seed=seed, max_eval=50, max_new_tokens=3)
        r = dict(r)
        r["N"] = N
        r["seed"] = seed
        rows_new.append(r)

        # checkpoint after each run
        df_running = pd.concat([df_partial, pd.DataFrame(rows_new)], ignore_index=True)
        df_running = df_running.drop_duplicates(["N","seed"], keep="last").sort_values(["N","seed"])
        df_running.to_csv("qwen3_boolq_icl_eval_RUNNING.csv", index=False)
        print("Saved checkpoint:", N, seed)
        cleanup_cuda()  # <-- after finishing the run


df_all = pd.read_csv("qwen3_boolq_icl_eval_RUNNING.csv")
df_all

Saved checkpoint: 64 0


Generating train split: 0 examples [00:00, ? examples/s]

Saved checkpoint: 64 1


Generating train split: 0 examples [00:00, ? examples/s]

Saved checkpoint: 64 2


,N,seed,n_eval,accuracy,invalid_frac,gen_time_s_mean,toks_per_sec_mean,prompt_tokens_mean,prompt_tokens_max,peak_mb_max,n_eval_total,n_eval_used,skipped,oom
0,16,0,200.0,0.885,0.0,5.264400,0.380604,2871.88,3450,7818.600586,NaN,NaN,NaN,NaN
1,16,1,200.0,0.885,0.0,5.389296,0.371676,2921.88,3500,7934.274414,NaN,NaN,NaN,NaN
2,16,2,200.0,0.890,0.0,4.388051,0.457151,2574.88,3153,7163.149902,NaN,NaN,NaN,NaN
3,32,0,200.0,0.890,0.0,7.266327,0.275787,5665.88,6244,15630.029300,NaN,NaN,NaN,NaN
4,32,1,200.0,0.880,0.0,6.275121,0.319204,5250.88,5829,14017.068360,NaN,NaN,NaN,NaN
5,32,2,200.0,0.890,0.0,6.859340,0.292135,5495.88,6074,14956.506350,NaN,NaN,NaN,NaN
6,64,0,NaN,0.880,0.0,7.734046,0.258876,5907.64,6060,15523.955566,50.0,50.0,0.0,0.0
7,64,1,NaN,0.880,0.0,7.380270,0.271269,5759.64,5912,14953.321289,50.0,50.0,0.0,0.0
8,64,2,NaN,0.900,0.0,7.306632,0.273997,5727.64,5880,14831.708008,50.0,50.0,0.0,0.0


In [ ]:
import pandas as pd

df_partial = pd.DataFrame(rows).sort_values(["N","seed"])
df_partial.to_csv("qwen3_boolq_icl_eval_PARTIAL.csv", index=False)
df_partial

NameError: name 'rows' is not defined

In [ ]:
df = pd.DataFrame(rows).sort_values(["N", "seed"])
df

,N,seed,n_eval,accuracy,invalid_frac,gen_time_s_mean,toks_per_sec_mean,prompt_tokens_mean,prompt_tokens_max,peak_mb_max
0,16,0,200,0.885,0.0,5.264400,0.380604,2871.88,3450,7818.600586
1,16,1,200,0.885,0.0,5.389296,0.371676,2921.88,3500,7934.274414
2,16,2,200,0.890,0.0,4.388051,0.457151,2574.88,3153,7163.149902


In [ ]:
df.to_csv("qwen3_boolq_icl_eval.csv", index=False)
print("Saved qwen3_boolq_icl_eval.csv")

Saved qwen3_boolq_icl_eval.csv
